# Meta 2.1 Aplicar técnicas de limpieza de datos usando Pandas
#### Cuevas Cortes Viridiana 2203301


In [16]:
# Cargar dataset
import pandas as pd
data=pd.read_csv("datasets/dirty_cafe_sales.csv")

print(data.shape)


(100, 8)


In [17]:
# Exploración inicial
data.info()
print("Dato nulos")
print(data.isnull().sum())

print("Dato duplicados")
print(data.duplicated().sum())



<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    100 non-null    str  
 1   Item              95 non-null     str  
 2   Quantity          99 non-null     str  
 3   Price Per Unit    97 non-null     str  
 4   Total Spent       98 non-null     str  
 5   Payment Method    69 non-null     str  
 6   Location          77 non-null     str  
 7   Transaction Date  99 non-null     str  
dtypes: str(8)
memory usage: 6.4 KB
Dato nulos
Transaction ID       0
Item                 5
Quantity             1
Price Per Unit       3
Total Spent          2
Payment Method      31
Location            23
Transaction Date     1
dtype: int64
Dato duplicados
0


In [18]:
# Ver las primeras 5 filas para entender la estructura
print(data.head())

  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-04-27  
4  In-store       2023-06-11  


In [19]:
#Conversion de tipo de dato
#Item: Eliminar espacios en blanco y convertir a minúsculas para consistencia
data['Item'] = data['Item'].str.strip()  
data['Item'] = data['Item'].str.lower() 

#Quantity: Convertir a numérico
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')


# Price Per Unit
data['Price Per Unit'] = pd.to_numeric(data['Price Per Unit'], errors='coerce')


#Total Spent: Convertir a numérico
data['Total Spent'] = pd.to_numeric(data['Total Spent'], errors='coerce')

#Transaction Date: Convertir a datetime
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], errors='coerce')


In [20]:
# Limpieza de Item  
# Convertir 'unknown' y 'error' a NaN
data['Item'] = data['Item'].replace('unknown', pd.NA)
data['Item'] = data['Item'].replace('error', pd.NA)


# Eliminar las filas que tengan NaN en la columna 'Item'
data.dropna(subset=['Item'], inplace=True)

print("Nulos restantes en 'Item':", data['Price Per Unit'].isnull().sum())


data.Item.sample(10)

print(data.shape)




Nulos restantes en 'Item': 4
(85, 8)


In [21]:
#Limpieza de Quantity 

# Rellenamos Quantity con la mediana 
mediana_qty = data['Quantity'].median()
data['Quantity'] = data['Quantity'].fillna(mediana_qty)

#Pasar Quantity a enteros
data['Quantity'] = data['Quantity'].astype(int)

print("Nulos restantes en 'Quantity':", data['Quantity'].isnull().sum())


data.Quantity.sample(10)



Nulos restantes en 'Quantity': 0


38    4
70    2
41    4
24    5
82    2
15    3
2     4
92    5
55    4
84    4
Name: Quantity, dtype: int64

In [22]:
#Calcular Price Per Unit para las filas donde falta, usando Total Spent y Quantity
data['Price Per Unit'] = data['Price Per Unit'].fillna(data['Total Spent'] / data['Quantity'])

# Eliminar las filas que tengan NaN en la columna 'Price Per Unit' después del cálculo
data.dropna(subset=['Price Per Unit'], inplace=True)

print("Nulos restantes en 'Price Per Unit':", data['Price Per Unit'].isnull().sum())

data['Price Per Unit'].sample(5)




Nulos restantes en 'Price Per Unit': 0


21    4.0
26    1.0
35    4.0
0     2.0
29    3.0
Name: Price Per Unit, dtype: float64

In [23]:
#Limpieza de Total Spent
# Calcular Total Spent para las filas donde falta, usando Quantity y Price Per Unit
data['Total Spent'] = data['Total Spent'].fillna(data['Quantity'] * data['Price Per Unit'])

print("Nulos restantes en 'Total Spent':", data['Total Spent'].isnull().sum())

data['Total Spent'].sample(5)




Nulos restantes en 'Total Spent': 0


27    12.0
75     2.0
48    15.0
9     20.0
12     8.0
Name: Total Spent, dtype: float64

In [24]:
#Limpieza de Payment Method
data['Payment Method'] = data['Payment Method'].str.strip()  # Eliminar espacios

# Cambiar 'UNKNOWN' y 'ERROR' como nulo
data['Payment Method'] = data['Payment Method'].replace(['UNKNOWN', 'ERROR'], pd.NA)

# Rellenar todos los nulos con "No Especificado"
data['Payment Method'] = data['Payment Method'].fillna('No Especificado')

print(data['Payment Method'].value_counts())

Payment Method
No Especificado    28
Cash               25
Credit Card        22
Digital Wallet      9
Name: count, dtype: int64


In [25]:
#Limpieza de Location
data['Location'] = data['Location'].str.strip()  # Eliminar espacios

# Cambiar 'UNKNOWN' y 'ERROR' como nulo
data['Location'] = data['Location'].replace(['UNKNOWN', 'ERROR'], pd.NA)

# Rellenar los nulos con "No Especificado"
data['Location'] = data['Location'].fillna('No Especificado')

print(data['Location'].value_counts())


Location
No Especificado    30
In-store           28
Takeaway           26
Name: count, dtype: int64


In [26]:
# Verificar cuántos nulos tenemos ahora (debería mostrar la suma de todos los errores)
print(f"Nulos en fecha antes de limpiar: {data['Transaction Date'].isnull().sum()}")

# Eliminar las filas sin fecha
data.dropna(subset=['Transaction Date'], inplace=True)

data['Transaction Date'].sample(5)



print("Dato nulos")
print(data.isnull().sum())

print("Dato duplicados")
print(data.duplicated().sum())


Nulos en fecha antes de limpiar: 3
Dato nulos
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64
Dato duplicados
0


In [27]:
# One-Hot Encoding para 'Item' y 'Payment Method'
# Esto creará columnas nuevas como: Item_Coffee, Item_Cake, Payment Method_Cash, etc.
data_encoded = pd.get_dummies(data, columns=['Item', 'Payment Method'])

print(data_encoded.filter(like='Item').head())
print(data_encoded.filter(like='Payment').head())

   Item_cake  Item_coffee  Item_cookie  Item_juice  Item_salad  Item_sandwich  \
0      False         True        False       False       False          False   
1       True        False        False       False       False          False   
2      False        False         True       False       False          False   
3      False        False        False       False        True          False   
4      False         True        False       False       False          False   

   Item_smoothie  Item_tea  
0          False     False  
1          False     False  
2          False     False  
3          False     False  
4          False     False  
   Payment Method_Cash  Payment Method_Credit Card  \
0                False                        True   
1                 True                       False   
2                False                        True   
3                False                       False   
4                False                       False   

   Payment Met

In [28]:
# Label Encoding para 'Location'
data['Location_Code'] = data['Location'].astype('category').cat.codes

# Verificamos la conversión
print(data[['Location', 'Location_Code']].sample(5))

           Location  Location_Code
68         In-store              0
73  No Especificado              1
3   No Especificado              1
4          In-store              0
98         Takeaway              2
